# MICS DDML analysis

The notebook is deliberately ordered in three stages:

1. **Pre-estimation diagnostics:** outcome and treatment coding, PSU structure, observed support by country and PSU, grouped-fold audits, and raw out-of-fold propensity diagnostics.
2. **Causal estimation:** grouped DDML/AIPW estimates using the convex Super Learner.
3. **Post-estimation robustness:** support-restricted estimates and leave-one-country-out analysis.

No causal effect is estimated before the diagnostic checkpoint. Estimating the raw propensity nuisance is part of the diagnostic stage, not a causal-effect estimate.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
from IPython.display import display

from mics_ddml_grouped_sl import (
    LEVELS,
    analysis_frame,
    contrast_results,
    eligible_countries,
    fit_apos,
    fit_irm,
    grouped_sample_splitting,
    irm_result,
    learner_library,
    prepare_sample,
    propensity_summary,
    psu_structure,
    raw_oof_propensities,
    support_by_country,
)

ROOT = Path("../../").resolve()
DATA = ROOT / "Data" / "3. Final"
OUT = ROOT / "Output"
FIGS = ROOT / "Figures"
MODELS = OUT / "models" / "grouped_convex_sl"
SUPPORT_MODELS = MODELS / "support_restricted"
LOCO_MODELS = MODELS / "loco"

for folder in [OUT, FIGS, MODELS, SUPPORT_MODELS, LOCO_MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

HH_FILE = DATA / "MASTER_MICS_FINAL.dta"
U5_FILE = DATA / "MASTER_MICS_FINAL_U5.dta"

SEED = 42
FOLDS = 5
IRM_REPS = 3
APOS_REPS = 1
ROBUSTNESS_REPS = 1
LOCO_REPS = 1
TRIM = 0.01
CPU = os.cpu_count() or 1
WORKERS = max(1, CPU - 1)

# This switch creates an explicit stopping point after all diagnostics.
RUN_CAUSAL_ESTIMATION = True
RUN_SUPPORT_ROBUSTNESS = True
RUN_LOCO = True

SL_G, SL_M = learner_library(seed=SEED, inner_folds=3)
display(pd.Series({
    "Folds": FOLDS,
    "IRM repetitions": IRM_REPS,
    "APOS repetitions": APOS_REPS,
    "Propensity clipping": TRIM,
    "Available CPUs": CPU,
}))


# Stage I. Pre-estimation diagnostics

## 2. Load and clean the samples

The treatment names follow the Stata descriptions:

| Code | Treatment category |
|---:|---|
| 0 | No treatment |
| 1 | Boiling |
| 2 | Chlorination/tablets |
| 3 | Straining/settling |

Recognized methods take precedence when multiple methods are reported. `Other-only` observations are dropped rather than recoded as untreated. Solar treatment remains a recognized treatment in the binary any-treatment comparison but is not a separate APOS category.


In [ ]:
hh_raw, _ = pyreadstat.read_dta(HH_FILE)
u5_raw, _ = pyreadstat.read_dta(U5_FILE)

hh, hh_cleaning = prepare_sample(hh_raw, child=False)
u5, u5_cleaning = prepare_sample(u5_raw, child=True)

cleaning_table = pd.DataFrame([
    {"sample": "HH", **hh_cleaning},
    {"sample": "U5", **u5_cleaning},
])
cleaning_table.to_csv(OUT / "cleaning_and_identifier_diagnostics.csv", index=False)
display(cleaning_table)


## 3. Outcome and treatment checks

`SomeRiskHome` equals one for any detectable E. coli and includes every `VeryHighRiskHome=1` observation. `VeryHighRiskHome` identifies concentrations above 100 CFU/100 mL. The outcomes are nested, but both models use the same complete household sample.


In [ ]:
assert not (hh["VeryHighRiskHome"].eq(1) & ~hh["SomeRiskHome"].eq(1)).any()
assert hh["SomeRiskHome"].isna().equals(hh["VeryHighRiskHome"].isna())

def treatment_frequency(df, sample_name):
    categorical = (
        df.loc[df["treat_cat"].isin(LEVELS), "treat_cat"]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("treatment_code")
        .reset_index(name="observations")
    )
    categorical["sample"] = sample_name
    categorical["treatment_category"] = categorical["treatment_code"].map(LEVELS)
    categorical["share"] = categorical["observations"] / categorical["observations"].sum()
    return categorical[["sample", "treatment_code", "treatment_category", "observations", "share"]]

treatment_frequencies = pd.concat([
    treatment_frequency(hh, "HH"),
    treatment_frequency(u5, "U5"),
], ignore_index=True)
treatment_frequencies.to_csv(OUT / "treatment_category_frequencies.csv", index=False)
display(treatment_frequencies)


## 4. PSU and household structure

The table reports the number of PSU and the mean, median, upper tail, and maximum number of observations per PSU. Country is combined with the raw PSU code only when the same PSU code appears in more than one country.


In [ ]:
psu_table = pd.DataFrame([
    psu_structure(hh, "HH"),
    psu_structure(u5, "U5"),
])
psu_table.to_csv(OUT / "psu_structure_summary.csv", index=False)
display(psu_table)


## 5. Observed support by country and PSU

For each comparison, the basic condition is that both `No treatment` and the relevant treatment category occur in the country. The stricter diagnostic asks whether each category occurs in at least two distinct PSU. We do **not** require every individual PSU to contain both categories.


In [ ]:
hh_support_detail, hh_support_summary = support_by_country(hh, "HH")
u5_support_detail, u5_support_summary = support_by_country(u5, "U5")

support_detail = pd.concat([hh_support_detail, u5_support_detail], ignore_index=True)
support_summary = pd.concat([hh_support_summary, u5_support_summary], ignore_index=True)

support_detail.to_csv(OUT / "positivity_support_by_country.csv", index=False)
support_summary.to_csv(OUT / "positivity_support_summary.csv", index=False)
display(support_summary)


## 6. Build the analysis samples

The two household outcomes must use identical observation indices. APOS excludes solar-only observations because solar is not one of the four method-specific categories, while the binary IRM retains solar as a recognized treatment.


In [ ]:
hh_some_irm, hh_irm_x = analysis_frame(hh, "SomeRiskHome", "water_treatment", child=False)
hh_vhigh_irm, hh_vhigh_irm_x = analysis_frame(hh, "VeryHighRiskHome", "water_treatment", child=False)
hh_some_apos, hh_apos_x = analysis_frame(
    hh, "SomeRiskHome", "treat_cat", child=False, allowed_levels=list(LEVELS)
)
hh_vhigh_apos, hh_vhigh_apos_x = analysis_frame(
    hh, "VeryHighRiskHome", "treat_cat", child=False, allowed_levels=list(LEVELS)
)
u5_irm, u5_irm_x = analysis_frame(u5, "diarrhea", "water_treatment", child=True)
u5_apos, u5_apos_x = analysis_frame(
    u5, "diarrhea", "treat_cat", child=True, allowed_levels=list(LEVELS)
)

assert hh_some_irm["_row_id"].equals(hh_vhigh_irm["_row_id"])
assert hh_some_apos["_row_id"].equals(hh_vhigh_apos["_row_id"])
assert hh_irm_x == hh_vhigh_irm_x
assert hh_apos_x == hh_vhigh_apos_x

sample_table = pd.DataFrame([
    {"sample": "HH IRM", "N": len(hh_some_irm), "PSUs": hh_some_irm["_psu_id"].nunique()},
    {"sample": "HH APOS", "N": len(hh_some_apos), "PSUs": hh_some_apos["_psu_id"].nunique()},
    {"sample": "U5 IRM", "N": len(u5_irm), "PSUs": u5_irm["_psu_id"].nunique()},
    {"sample": "U5 APOS", "N": len(u5_apos), "PSUs": u5_apos["_psu_id"].nunique()},
])
display(sample_table)


## 7. Construct and audit PSU-grouped folds

A PSU is assigned wholly to training or test. The audit also verifies that no household leaks across the split, each observation is tested exactly once per repetition, and every training fold contains every required treatment category.


In [ ]:
hh_irm_smpls, hh_irm_cluster, hh_irm_audit = grouped_sample_splitting(
    hh_some_irm, "water_treatment", IRM_REPS, FOLDS, SEED
)
hh_apos_smpls, hh_apos_cluster, hh_apos_audit = grouped_sample_splitting(
    hh_some_apos, "treat_cat", APOS_REPS, FOLDS, SEED
)
u5_irm_smpls, u5_irm_cluster, u5_irm_audit = grouped_sample_splitting(
    u5_irm, "water_treatment", IRM_REPS, FOLDS, SEED
)
u5_apos_smpls, u5_apos_cluster, u5_apos_audit = grouped_sample_splitting(
    u5_apos, "treat_cat", APOS_REPS, FOLDS, SEED
)

fold_audit = pd.concat([
    hh_irm_audit.assign(sample="HH", model="IRM"),
    hh_apos_audit.assign(sample="HH", model="APOS"),
    u5_irm_audit.assign(sample="U5", model="IRM"),
    u5_apos_audit.assign(sample="U5", model="APOS"),
], ignore_index=True)
fold_audit.to_csv(OUT / "fold_balance_grouped_convex_sl.csv", index=False)
display(fold_audit)


## 8. Raw out-of-fold propensity diagnostics

This stage estimates only the treatment nuisance functions. Raw OOF propensities are used to diagnose positivity; clipped propensities are retained separately for the AIPW score. The household table is computed once because both E. coli outcomes use the same sample and treatment model.


In [ ]:
hh_oof, hh_sl_weights = raw_oof_propensities(
    SL_M, hh_some_apos, hh_apos_x, hh_apos_smpls, TRIM, "HH"
)
u5_oof, u5_sl_weights = raw_oof_propensities(
    SL_M, u5_apos, u5_apos_x, u5_apos_smpls, TRIM, "U5"
)

propensity_oof = pd.concat([hh_oof, u5_oof], ignore_index=True)
propensity_table = propensity_summary(propensity_oof)
sl_weights = pd.concat([hh_sl_weights, u5_sl_weights], ignore_index=True)

propensity_oof.to_csv(OUT / "propensity_oof_raw_and_clipped.csv", index=False)
propensity_table.to_csv(OUT / "propensity_oof_summary.csv", index=False)
sl_weights.to_csv(OUT / "super_learner_propensity_weights.csv", index=False)
display(propensity_table)


In [ ]:
def ecdf(values):
    x = np.sort(np.asarray(values))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

for sample_name, oof in [("HH", hh_oof), ("U5", u5_oof)]:
    for level in [1, 2, 3]:
        plot_data = oof.loc[
            oof["treatment_code"].eq(level)
            & oof["observed_category"].isin([0, level])
        ]
        plt.figure(figsize=(7, 4.5))
        for observed_level in [0, level]:
            values = plot_data.loc[
                plot_data["observed_category"].eq(observed_level),
                "propensity_raw",
            ]
            x, y = ecdf(values)
            plt.plot(x, y, label=LEVELS[observed_level])
        for threshold in [0.01, 0.025, 0.05]:
            plt.axvline(threshold, linestyle="--", linewidth=0.8)
        plt.xlabel(f"Raw OOF P({LEVELS[level]} | X)")
        plt.ylabel("Empirical cumulative probability")
        plt.title(f"{sample_name}: {LEVELS[level]} vs No treatment")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            FIGS / f"positivity_ecdf_{sample_name.lower()}_{level}.png",
            dpi=300, bbox_inches="tight"
        )
        plt.show()


## 9. Diagnostic checkpoint

At this point, and **before any causal effect is estimated**, review:

1. outcome nesting and identical HH samples;
2. number and size distribution of PSU;
3. treatment counts and the number of `Other-only` observations dropped;
4. countries with both comparison categories present;
5. countries with at least two PSU in each comparison category;
6. fold balance and zero PSU/household leakage;
7. raw OOF propensity percentiles and tails.

The AIPW score is doubly robust, but double robustness does not create support where a treatment category is absent. Clipping stabilizes the score; it does not repair structural nonpositivity. Set `RUN_CAUSAL_ESTIMATION=False` in the setup cell to stop the notebook here.


In [ ]:
diagnostic_manifest = {
    "cleaning": str(OUT / "cleaning_and_identifier_diagnostics.csv"),
    "treatment_frequencies": str(OUT / "treatment_category_frequencies.csv"),
    "PSU_structure": str(OUT / "psu_structure_summary.csv"),
    "support_summary": str(OUT / "positivity_support_summary.csv"),
    "support_by_country": str(OUT / "positivity_support_by_country.csv"),
    "fold_audit": str(OUT / "fold_balance_grouped_convex_sl.csv"),
    "raw_propensities": str(OUT / "propensity_oof_raw_and_clipped.csv"),
    "propensity_summary": str(OUT / "propensity_oof_summary.csv"),
}
(OUT / "pre_estimation_diagnostic_manifest.json").write_text(
    json.dumps(diagnostic_manifest, indent=2), encoding="utf-8"
)
display(pd.Series(diagnostic_manifest, name="path"))


# Stage II. Causal estimation

Only the convex Super Learner is treated as the principal nuisance specification. Individual learner estimates belong in appendix robustness checks and are not interpreted as independent causal estimators.


In [ ]:
main_results = []
main_models = {}

if RUN_CAUSAL_ESTIMATION:
    specs = [
        ("HH", "SomeRiskHome", hh_some_irm, hh_irm_x, hh_irm_smpls, hh_irm_cluster,
         hh_some_apos, hh_apos_x, hh_apos_smpls, hh_apos_cluster),
        ("HH", "VeryHighRiskHome", hh_vhigh_irm, hh_vhigh_irm_x, hh_irm_smpls, hh_irm_cluster,
         hh_vhigh_apos, hh_vhigh_apos_x, hh_apos_smpls, hh_apos_cluster),
        ("U5", "diarrhea", u5_irm, u5_irm_x, u5_irm_smpls, u5_irm_cluster,
         u5_apos, u5_apos_x, u5_apos_smpls, u5_apos_cluster),
    ]

    for (dataset, outcome, irm_frame, irm_x, irm_smpls, irm_cluster,
         apos_frame, apos_x, apos_smpls, apos_cluster) in specs:
        irm_model = fit_irm(
            irm_frame, irm_x, outcome, "water_treatment",
            irm_smpls, irm_cluster, SL_G, SL_M, TRIM,
            MODELS / f"irm_{dataset}_{outcome}.pkl",
            n_jobs_cv=min(WORKERS, FOLDS),
        )
        apos_model, contrast = fit_apos(
            apos_frame, apos_x, outcome, "treat_cat",
            apos_smpls, apos_cluster, SL_G, SL_M, TRIM,
            MODELS / f"apos_{dataset}_{outcome}.pkl",
            n_jobs_cv=min(WORKERS, FOLDS),
            n_jobs_models=min(WORKERS, len(LEVELS)),
        )
        main_results.append(irm_result(irm_model, dataset, outcome))
        main_results.extend(contrast_results(contrast, dataset, outcome))
        main_models[(dataset, outcome, "IRM")] = irm_model
        main_models[(dataset, outcome, "APOS")] = apos_model

    main_results = pd.DataFrame(main_results)
    main_results.to_csv(OUT / "results_main_grouped_convex_sl.csv", index=False)
    display(main_results)
else:
    print("Diagnostics complete. Causal estimation is disabled.")


# Stage III. Post-estimation robustness

## 10. Support-restricted comparisons

These estimates are robustness checks because restricting countries changes the target population. Each method is compared only with `No treatment`; absence of another treatment category does not remove a country from an unrelated comparison.


In [ ]:
support_results = []

if RUN_CAUSAL_ESTIMATION and RUN_SUPPORT_ROBUSTNESS:
    raw_specs = [
        ("HH", hh, hh_support_detail, "SomeRiskHome", False),
        ("HH", hh, hh_support_detail, "VeryHighRiskHome", False),
        ("U5", u5, u5_support_detail, "diarrhea", True),
    ]

    for dataset, raw_df, detail, outcome, child in raw_specs:
        for level in [1, 2, 3]:
            for rule, minimum_psu in [
                ("Both categories present", 1),
                (">=2 PSUs in each category", 2),
            ]:
                countries = eligible_countries(detail, level, minimum_psu)
                restricted = raw_df.loc[
                    raw_df["country_cat"].isin(countries)
                    & raw_df["treat_cat"].isin([0, level])
                ].copy()
                restricted["pair_treatment"] = restricted["treat_cat"].eq(level).astype("int8")
                frame, x_cols = analysis_frame(
                    restricted, outcome, "pair_treatment", child=child, allowed_levels=[0, 1]
                )
                if frame["pair_treatment"].nunique() < 2 or frame["_psu_id"].nunique() < FOLDS:
                    continue
                smpls, clusters, _ = grouped_sample_splitting(
                    frame, "pair_treatment", ROBUSTNESS_REPS, FOLDS, SEED
                )
                model = fit_irm(
                    frame, x_cols, outcome, "pair_treatment", smpls, clusters,
                    SL_G, SL_M, TRIM,
                    SUPPORT_MODELS / f"{dataset}_{outcome}_{level}_{minimum_psu}psu.pkl",
                    n_jobs_cv=min(WORKERS, FOLDS),
                )
                result = irm_result(model, dataset, outcome)
                result.update({
                    "method": "Support-restricted IRM",
                    "comparison": f"{LEVELS[level]} vs No treatment",
                    "support_rule": rule,
                    "countries": len(countries),
                    "N": len(frame),
                    "PSUs": frame["_psu_id"].nunique(),
                })
                support_results.append(result)

    support_results = pd.DataFrame(support_results)
    support_results.to_csv(OUT / "results_support_restricted.csv", index=False)
    display(support_results)


## 11. Leave-one-country-out

LOCO measures influence of individual countries; it is not a substitute for the positivity diagnostics. Country dummies, grouped folds, and nuisance functions are rebuilt after each country is removed.


In [ ]:
loco_results = []

if RUN_CAUSAL_ESTIMATION and RUN_LOCO:
    raw_specs = [
        ("HH", hh, "SomeRiskHome", False),
        ("HH", hh, "VeryHighRiskHome", False),
        ("U5", u5, "diarrhea", True),
    ]

    for dataset, raw_df, outcome, child in raw_specs:
        for country in sorted(raw_df["country_cat"].dropna().unique()):
            restricted = raw_df.loc[~raw_df["country_cat"].eq(country)].copy()
            irm_frame, irm_x = analysis_frame(restricted, outcome, "water_treatment", child=child)
            apos_frame, apos_x = analysis_frame(
                restricted, outcome, "treat_cat", child=child, allowed_levels=list(LEVELS)
            )
            if irm_frame["water_treatment"].nunique() < 2:
                continue
            if set(apos_frame["treat_cat"].unique()) != set(LEVELS):
                continue

            irm_smpls, irm_clusters, _ = grouped_sample_splitting(
                irm_frame, "water_treatment", LOCO_REPS, FOLDS, SEED
            )
            apos_smpls, apos_clusters, _ = grouped_sample_splitting(
                apos_frame, "treat_cat", LOCO_REPS, FOLDS, SEED
            )
            tag = str(country).replace("/", "_").replace(" ", "_")
            irm_model = fit_irm(
                irm_frame, irm_x, outcome, "water_treatment", irm_smpls, irm_clusters,
                SL_G, SL_M, TRIM, LOCO_MODELS / f"irm_{dataset}_{outcome}_{tag}.pkl",
                n_jobs_cv=min(WORKERS, FOLDS),
            )
            apos_model, contrast = fit_apos(
                apos_frame, apos_x, outcome, "treat_cat", apos_smpls, apos_clusters,
                SL_G, SL_M, TRIM, LOCO_MODELS / f"apos_{dataset}_{outcome}_{tag}.pkl",
                n_jobs_cv=min(WORKERS, FOLDS),
                n_jobs_models=min(WORKERS, len(LEVELS)),
            )
            rows = [irm_result(irm_model, dataset, outcome)]
            rows.extend(contrast_results(contrast, dataset, outcome))
            for row in rows:
                row.update({
                    "excluded_country": country,
                    "remaining_N_IRM": len(irm_frame),
                    "remaining_PSU_IRM": irm_frame["_psu_id"].nunique(),
                    "remaining_N_APOS": len(apos_frame),
                    "remaining_PSU_APOS": apos_frame["_psu_id"].nunique(),
                })
                loco_results.append(row)
            pd.DataFrame(loco_results).to_csv(
                OUT / "results_leave_one_country_out.csv", index=False
            )

    loco_results = pd.DataFrame(loco_results)
    full = main_results[["dataset", "outcome", "comparison", "coef", "se"]].rename(
        columns={"coef": "full_coef", "se": "full_se"}
    )
    loco_results = loco_results.merge(
        full, on=["dataset", "outcome", "comparison"], how="left"
    )
    loco_results["change_from_full"] = loco_results["coef"] - loco_results["full_coef"]
    loco_results["change_in_full_SE"] = loco_results["change_from_full"] / loco_results["full_se"]
    loco_results.to_csv(OUT / "results_leave_one_country_out.csv", index=False)
    display(loco_results.head())


## 12. Final output manifest


In [ ]:
final_manifest = {
    **diagnostic_manifest,
    "main_results": str(OUT / "results_main_grouped_convex_sl.csv"),
    "support_restricted_results": str(OUT / "results_support_restricted.csv"),
    "LOCO_results": str(OUT / "results_leave_one_country_out.csv"),
}
(OUT / "grouped_convex_sl_manifest.json").write_text(
    json.dumps(final_manifest, indent=2), encoding="utf-8"
)
display(pd.Series(final_manifest, name="path"))
